# Query one IDC collection and download a few series

Uses the Imaging Data Commons index (`idc-index`) against the `rider_pilot` collection, then downloads the three smallest CT series.

In [1]:
import sys

# Install into this notebook's interpreter (the project .venv), not whichever pip is first on PATH.
!{sys.executable} -m pip install -q idc-index
print(sys.executable)


/Users/taiaburrahman/Desktop/git/idc-index/.venv/bin/python


In [2]:
from idc_index import IDCClient

client = IDCClient.client()
client

In [3]:
COLLECTION = "rider_pilot"

summary = client.sql_query(f"""
SELECT
    collection_id,
    COUNT(DISTINCT PatientID) AS patients,
    COUNT(DISTINCT StudyInstanceUID) AS studies,
    COUNT(DISTINCT SeriesInstanceUID) AS series,
    ROUND(SUM(series_size_MB), 1) AS total_size_MB
FROM index
WHERE collection_id = '{COLLECTION}'
GROUP BY collection_id
""")
summary

,collection_id,patients,studies,series,total_size_MB
0,rider_pilot,8,154,774,11259.2


In [4]:
series = client.sql_query(f"""
SELECT
    collection_id,
    PatientID,
    StudyInstanceUID,
    SeriesInstanceUID,
    Modality,
    SeriesDescription,
    ROUND(series_size_MB, 2) AS series_size_MB
FROM index
WHERE collection_id = '{COLLECTION}'
  AND Modality = 'CT'
  AND series_size_MB > 0
ORDER BY series_size_MB ASC
LIMIT 3
""")
series

,collection_id,PatientID,StudyInstanceUID,SeriesInstanceUID,Modality,SeriesDescription,series_size_MB
0,rider_pilot,1.3.6.1.4.1.9328.50.1.0109,1.3.6.1.4.1.9328.50.1.53671,1.3.6.1.4.1.9328.50.1.53734,CT,Smart Prep Series,0.53
1,rider_pilot,1.3.6.1.4.1.9328.50.1.0109,1.3.6.1.4.1.9328.50.1.54652,1.3.6.1.4.1.9328.50.1.54657,CT,Smart Prep Series,0.53
2,rider_pilot,1.3.6.1.4.1.9328.50.1.0121,1.3.6.1.4.1.9328.50.1.69732,1.3.6.1.4.1.9328.50.1.69736,CT,NaN,0.53


In [5]:
from pathlib import Path

download_dir = Path("downloads") / COLLECTION
download_dir.mkdir(parents=True, exist_ok=True)

client.download_dicom_series(
    seriesInstanceUID=series["SeriesInstanceUID"].tolist(),
    downloadDir=str(download_dir),
    quiet=False,
    show_progress_bar=True,
)

files = sorted(p for p in download_dir.rglob("*") if p.is_file())
print(f"Downloaded {len(files)} files into {download_dir.resolve()}")
for path in files[:12]:
    print(f"  {path.relative_to(download_dir)}  ({path.stat().st_size / 1_048_576:.2f} MB)")
if len(files) > 12:
    print(f"  ... and {len(files) - 12} more")

2026-10-28 20:02:59,202 - Disk size needed: 1.58 MB


2026-10-28 20:02:59,202 - Disk size available: 5.92 GB


2026-10-28 20:03:02,338 - Not using s5cmd sync as the destination folder is empty or sync or progress bar is not requested


2026-10-28 20:03:02,354 - Initial size of the directory: 526056 bytes


2026-10-28 20:03:02,355 - Approximate size of the files that need to be downloaded: 1.58 MB



2026-10-28 20:03:04,543 - Successfully downloaded files to /Users/taiaburrahman/Desktop/git/idc-index/downloads/rider_pilot


cp s3://idc-open-data/247c32f9-efc9-4b8f-b3b2-3a9fe9f802e9/dbb51489-99d0-4d01-9ac3-9830b7468cb5.dcm /Users/taiaburrahman/Desktop/git/idc-index/downloads/rider_pilot/rider_pilot/1.3.6.1.4.1.9328.50.1.0121/1.3.6.1.4.1.9328.50.1.69732/CT_1.3.6.1.4.1.9328.50.1.69736/dbb51489-99d0-4d01-9ac3-9830b7468cb5.dcm
cp s3://idc-open-data/ec673959-b31f-47bb-acac-3a93fa8a2e28/efb0f0cf-fb7a-4769-9bc2-6dd0fc97f17a.dcm /Users/taiaburrahman/Desktop/git/idc-index/downloads/rider_pilot/rider_pilot/1.3.6.1.4.1.9328.50.1.0109/1.3.6.1.4.1.9328.50.1.54652/CT_1.3.6.1.4.1.9328.50.1.54657/efb0f0cf-fb7a-4769-9bc2-6dd0fc97f17a.dcm
cp s3://idc-open-data/7a503dc0-b864-4064-b74f-e9e32b1e1c24/fafa7d2d-7829-4704-ba5a-a4b430fcdc5a.dcm /Users/taiaburrahman/Desktop/git/idc-index/downloads/rider_pilot/rider_pilot/1.3.6.1.4.1.9328.50.1.0109/1.3.6.1.4.1.9328.50.1.53671/CT_1.3.6.1.4.1.9328.50.1.53734/fafa7d2d-7829-4704-ba5a-a4b430fcdc5a.dcm


Downloaded 3 files into /Users/taiaburrahman/Desktop/git/idc-index/downloads/rider_pilot
  rider_pilot/1.3.6.1.4.1.9328.50.1.0109/1.3.6.1.4.1.9328.50.1.53671/CT_1.3.6.1.4.1.9328.50.1.53734/fafa7d2d-7829-4704-ba5a-a4b430fcdc5a.dcm  (0.50 MB)
  rider_pilot/1.3.6.1.4.1.9328.50.1.0109/1.3.6.1.4.1.9328.50.1.54652/CT_1.3.6.1.4.1.9328.50.1.54657/efb0f0cf-fb7a-4769-9bc2-6dd0fc97f17a.dcm  (0.50 MB)
  rider_pilot/1.3.6.1.4.1.9328.50.1.0121/1.3.6.1.4.1.9328.50.1.69732/CT_1.3.6.1.4.1.9328.50.1.69736/dbb51489-99d0-4d01-9ac3-9830b7468cb5.dcm  (0.50 MB)
